In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BRONZE_HAZMAT_CLASS_FILES = [r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\hazmatclass-20260331T183130Z-3-001\hazmatclass\ams__hazmatclass_2018__202001290000_part_0.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\hazmatclass-20260331T184857Z-3-001\hazmatclass\ams__hazmatclass_2019__202001080000_part_0.csv']
SILVER_HAZMAT_CLASS_OUTPUT_PATH = r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\hazmat_class_silver.csv'

In [3]:
def load_hazmat_class_data(file_paths):
    dataframes = []
    for file in file_paths:
        df = pd.read_csv(file)
        dataframes.append(df)
    return pd.concat(dataframes, ignore_index=True)

hazmat_class_df = load_hazmat_class_data(BRONZE_HAZMAT_CLASS_FILES)
print("Hazmat class data loaded successfully. Shape:", hazmat_class_df.shape)

Hazmat class data loaded successfully. Shape: (35629, 4)


In [4]:
def normalize_hazmat_identifiers(df):
    if "container_number" in df.columns:
        df["container_number"] = (
            df["container_number"]
            .astype(str)
            .str.upper()
            .str.strip()
        )

        df.loc[
            df["container_number"].isin(["NC"]),
            "container_number",
        ] = pd.NA

    return df

hazmat_class_df = normalize_hazmat_identifiers(hazmat_class_df)
print("Hazmat identifiers normalized successfully.")

Hazmat identifiers normalized successfully.


In [6]:
def convert_hazmat_sequence(df):
    if "hazmat_sequence_number" in df.columns:
        df["hazmat_sequence_number"] = pd.to_numeric(
            df["hazmat_sequence_number"], errors="coerce", downcast="integer"
        )
    return df

hazmat_class_df = convert_hazmat_sequence(hazmat_class_df)
print("Hazmat sequence numbers converted successfully.")

Hazmat sequence numbers converted successfully.


In [7]:
def normalize_hazmat_classification(df):
    if "hazmat_classification" in df.columns:
        df["hazmat_classification"] = (
            df["hazmat_classification"]
            .astype(str)
            .str.upper()
            .str.strip()
        )

        df.loc[
            df["hazmat_classification"].isin(["", "NULL", "N/A"]),
            "hazmat_classification",
        ] = pd.NA

    return df

hazmat_class_df = normalize_hazmat_classification(hazmat_class_df)
print("Hazmat classifications normalized successfully.")

Hazmat classifications normalized successfully.


In [8]:
def remove_hazmat_duplicates(df):
    df.drop_duplicates(
        subset=["identifier",
                "container_number",
                "hazmat_sequence_number",
                "hazmat_classification"],
        keep="first",
        inplace=True
    )

remove_hazmat_duplicates(hazmat_class_df)
print("Duplicate rows removed successfully. Remaining shape:", hazmat_class_df.shape)

Duplicate rows removed successfully. Remaining shape: (35629, 4)


In [9]:
def build_silver_hazmat_class(df):
    silver_cols = ["identifier", "container_number", "hazmat_sequence_number", "hazmat_classification"]

    existing_cols = [c for c in silver_cols if c in df.columns]
    return df.loc[:, existing_cols]

silver_hazmat_class_df = build_silver_hazmat_class(hazmat_class_df)
print("Silver hazmat class dataframe created. Shape:", silver_hazmat_class_df.shape)

Silver hazmat class dataframe created. Shape: (35629, 4)


In [10]:
def save_silver_hazmat_class(df, output_path):
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_file, index=False)

save_silver_hazmat_class(silver_hazmat_class_df, SILVER_HAZMAT_CLASS_OUTPUT_PATH)
print(f"Silver hazmat class data saved to {SILVER_HAZMAT_CLASS_OUTPUT_PATH}.")

Silver hazmat class data saved to C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\hazmat_class_silver.csv.
